# Modeling comparison for fraud detection

Notebook này so sánh 4 thuật toán:
- Logistic Regression
- Decision Tree
- Bayesian (GaussianNB + SVD)
- ANN (MLP + SVD)

Dữ liệu đọc từ `artifacts/processed_csv/` do `fraud_detection_btl.ipynb` xuất (ví dụ **20 cột** feature sau bước chọn `selected_features_model`).

- `X_train.csv` / `y_train.csv`: train **gốc** (imbalanced).
- `X_train_balanced.csv` / `y_train_balanced.csv`: train sau **2.8 SMOTE**.

Biến `USE_BALANCED_TRAIN`:
- `False`: chỉ huấn luyện trên tập **imbalanced**.
- `True`: huấn luyện **song song** trên **imbalanced** và **balanced** (cùng 4 thuật toán), bảng kết quả có cột `train_variant` để so sánh.

Metrics: `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`, `accuracy` + confusion matrix + classification report.

Sau cell cuối: lưu thêm **mô hình đã fit** (`.pkl`, dùng `joblib`) trong `artifacts/modeling_results/trained_models/`.

In [18]:
from pathlib import Path
import joblib
import warnings
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import TruncatedSVD

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
TARGET = "is_fraud"


In [10]:
# Load dữ liệu đã export từ notebook tiền xử lý
# True: huấn luyện song song imbalanced + balanced (SMOTE); False: chỉ imbalanced
USE_BALANCED_TRAIN = True

ARTIFACT_DIR = Path("../artifacts") if Path.cwd().name == "notebooks" else Path("artifacts")
processed_dir = ARTIFACT_DIR / "processed_csv"

required_files = [
    "X_train.csv", "y_train.csv",
    "X_valid.csv", "X_test.csv", "y_valid.csv", "y_test.csv",
]
if USE_BALANCED_TRAIN:
    required_files += ["X_train_balanced.csv", "y_train_balanced.csv"]

missing = [f for f in required_files if not (processed_dir / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Thiếu file trong {processed_dir.resolve()}: {missing}. "
        "Chạy cell export cuối fraud_detection_btl.ipynb; nếu USE_BALANCED_TRAIN=True cần đã chạy 2.8.2 trước export."
    )

X_train = pd.read_csv(processed_dir / "X_train.csv")
y_train = pd.read_csv(processed_dir / "y_train.csv")[TARGET]

X_train_balanced = y_train_balanced = None
if USE_BALANCED_TRAIN:
    X_train_balanced = pd.read_csv(processed_dir / "X_train_balanced.csv")
    y_train_balanced = pd.read_csv(processed_dir / "y_train_balanced.csv")[TARGET]
    assert list(X_train.columns) == list(X_train_balanced.columns), "Schema train imbalanced vs balanced không khớp"

X_valid = pd.read_csv(processed_dir / "X_valid.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")
y_valid = pd.read_csv(processed_dir / "y_valid.csv")[TARGET]
y_test = pd.read_csv(processed_dir / "y_test.csv")[TARGET]

print("Loaded from:", processed_dir.resolve())
print("Train imbalanced:", X_train.shape, "fraud rate", y_train.mean())
if USE_BALANCED_TRAIN:
    print("Train balanced:  ", X_train_balanced.shape, "fraud rate", y_train_balanced.mean())
print("Valid:", X_valid.shape, "| Test:", X_test.shape)
print("Fraud rate valid/test:", y_valid.mean(), y_test.mean())


Loaded from: D:\Project\DataMining-DataWarehouse\Source\artifacts\processed_csv
Train imbalanced: (1037340, 20) fraud rate 0.005753176393467908
Train balanced:   (2062744, 20) fraud rate 0.5
Valid: (259335, 20) | Test: (555719, 20)
Fraud rate valid/test: 0.005930553145545337 0.0038598644278853163


In [11]:
# Cột khớp với CSV từ fraud_detection_btl: object/category -> one-hot, còn lại -> numeric + scale
feature_columns = list(X_train.columns)
if TARGET in feature_columns:
    raise ValueError(f"X_train không được chứa cột target '{TARGET}'")

categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = [c for c in feature_columns if c not in categorical_features]

print(f"Số cột feature: {len(feature_columns)} (numeric={len(numeric_features)}, categorical={len(categorical_features)})")
print("Categorical:", categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

transformers = [("num", numeric_pipeline, numeric_features)]
if categorical_features:
    transformers.append(("cat", categorical_pipeline, categorical_features))

base_preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

# TruncatedSVD bắt buộc n_components <= số cột sau prep (ví dụ 20 feature số → không dùng được 60)
_probe_prep = clone(base_preprocessor)
_n_rows = min(20000, len(X_train))
_n_prep = _probe_prep.fit_transform(X_train.iloc[:_n_rows]).shape[1]
SVD_NB_COMPONENTS = max(1, min(60, _n_prep))
SVD_MLP_COMPONENTS = max(1, min(80, _n_prep))
print(f"Chiều sau preprocessor: {_n_prep} → TruncatedSVD: NB={SVD_NB_COMPONENTS}, MLP={SVD_MLP_COMPONENTS}")


Số cột feature: 20 (numeric=20, categorical=0)
Categorical: []
Chiều sau preprocessor: 20 → TruncatedSVD: NB=20, MLP=20


In [12]:
models = {
    "LogisticRegression": Pipeline([
        ("prep", base_preprocessor),
        ("clf", LogisticRegression(
            class_weight="balanced",
            max_iter=300,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "DecisionTree": Pipeline([
        ("prep", base_preprocessor),
        ("clf", DecisionTreeClassifier(
            max_depth=12,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Bayesian_GaussianNB": Pipeline([
        ("prep", base_preprocessor),
        ("svd", TruncatedSVD(n_components=SVD_NB_COMPONENTS, random_state=RANDOM_STATE)),
        ("clf", GaussianNB()),
    ]),
    "ANN_MLP": Pipeline([
        ("prep", base_preprocessor),
        ("svd", TruncatedSVD(n_components=SVD_MLP_COMPONENTS, random_state=RANDOM_STATE)),
        ("clf", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            early_stopping=True,
            max_iter=40,
            random_state=RANDOM_STATE,
        )),
    ]),
}

def evaluate_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
    }

train_jobs = [("imbalanced", X_train, y_train)]
if USE_BALANCED_TRAIN:
    train_jobs.append(("balanced_smote", X_train_balanced, y_train_balanced))

results = []
artifacts = {}

for train_variant, Xt, yt in train_jobs:
    for name, model_template in models.items():
        print(f"\n=== [{train_variant}] Training {name} ===")
        model = clone(model_template)
        model.fit(Xt, yt)

        valid_prob = model.predict_proba(X_valid)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        valid_metrics = evaluate_binary(y_valid, valid_prob, threshold=0.5)
        test_metrics = evaluate_binary(y_test, test_prob, threshold=0.5)

        row = {"train_variant": train_variant, "model": name}
        row.update({f"valid_{k}": v for k, v in valid_metrics.items()})
        row.update({f"test_{k}": v for k, v in test_metrics.items()})
        results.append(row)

        artifacts.setdefault(train_variant, {})[name] = {
            "fitted_model": model,
            "valid_prob": valid_prob,
            "test_prob": test_prob,
        }

sort_cols = ["valid_pr_auc", "valid_f1"]
if USE_BALANCED_TRAIN:
    sort_cols = ["train_variant"] + sort_cols
    asc = [True, False, False]
else:
    asc = [False, False]
results_df = pd.DataFrame(results).sort_values(sort_cols, ascending=asc)
results_df



=== [imbalanced] Training LogisticRegression ===

=== [imbalanced] Training DecisionTree ===

=== [imbalanced] Training Bayesian_GaussianNB ===

=== [imbalanced] Training ANN_MLP ===

=== [balanced_smote] Training LogisticRegression ===

=== [balanced_smote] Training DecisionTree ===

=== [balanced_smote] Training Bayesian_GaussianNB ===

=== [balanced_smote] Training ANN_MLP ===


,train_variant,model,valid_accuracy,valid_precision,valid_recall,valid_f1,valid_roc_auc,valid_pr_auc,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
7,balanced_smote,ANN_MLP,0.987059,0.299515,0.882965,0.447299,0.989774,0.825827,0.991134,0.285670,0.864336,0.429415,0.989801,0.788728
5,balanced_smote,DecisionTree,0.958023,0.115562,0.913524,0.205169,0.971640,0.655554,0.960624,0.080760,0.886247,0.148030,0.963101,0.548113
6,balanced_smote,Bayesian_GaussianNB,0.882623,0.035039,0.708062,0.066773,0.842829,0.230873,0.912582,0.029315,0.674126,0.056186,0.838394,0.174311
4,balanced_smote,LogisticRegression,0.916934,0.052883,0.769181,0.098963,0.881590,0.193885,0.963312,0.070935,0.703030,0.128867,0.865761,0.137340
3,imbalanced,ANN_MLP,0.998072,0.917203,0.741873,0.820273,0.992471,0.865215,0.998578,0.892754,0.717949,0.795866,0.991649,0.832713
1,imbalanced,DecisionTree,0.983026,0.252249,0.947984,0.398470,0.973154,0.831510,0.983337,0.179533,0.929138,0.300921,0.963208,0.775889
0,imbalanced,LogisticRegression,0.934972,0.068136,0.786086,0.125402,0.894212,0.205016,0.953610,0.059820,0.748718,0.110789,0.877053,0.143336
2,imbalanced,Bayesian_GaussianNB,0.976474,0.129807,0.520156,0.207765,0.867946,0.163571,0.974942,0.077838,0.506294,0.134932,0.854416,0.104965


In [13]:
# Bảng so sánh chính (validation + test); train_variant = imbalanced | balanced_smote
cols = [
    "train_variant", "model",
    "valid_precision", "valid_recall", "valid_f1", "valid_roc_auc", "valid_pr_auc",
    "test_precision", "test_recall", "test_f1", "test_roc_auc", "test_pr_auc",
]
display(results_df[cols].reset_index(drop=True))

if USE_BALANCED_TRAIN:
    for v in results_df["train_variant"].unique():
        sub = results_df[results_df["train_variant"] == v].sort_values(
            ["valid_pr_auc", "valid_f1"], ascending=False
        )
        best = sub.iloc[0]["model"]
        print(f"\nTốt nhất theo valid PR-AUC (train={v}): {best}")
else:
    sub = results_df.sort_values(["valid_pr_auc", "valid_f1"], ascending=False)
    print(f"\nMô hình tốt nhất theo valid PR-AUC: {sub.iloc[0]['model']}")


,train_variant,model,valid_precision,valid_recall,valid_f1,valid_roc_auc,valid_pr_auc,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,balanced_smote,ANN_MLP,0.299515,0.882965,0.447299,0.989774,0.825827,0.285670,0.864336,0.429415,0.989801,0.788728
1,balanced_smote,DecisionTree,0.115562,0.913524,0.205169,0.971640,0.655554,0.080760,0.886247,0.148030,0.963101,0.548113
2,balanced_smote,Bayesian_GaussianNB,0.035039,0.708062,0.066773,0.842829,0.230873,0.029315,0.674126,0.056186,0.838394,0.174311
3,balanced_smote,LogisticRegression,0.052883,0.769181,0.098963,0.881590,0.193885,0.070935,0.703030,0.128867,0.865761,0.137340
4,imbalanced,ANN_MLP,0.917203,0.741873,0.820273,0.992471,0.865215,0.892754,0.717949,0.795866,0.991649,0.832713
5,imbalanced,DecisionTree,0.252249,0.947984,0.398470,0.973154,0.831510,0.179533,0.929138,0.300921,0.963208,0.775889
6,imbalanced,LogisticRegression,0.068136,0.786086,0.125402,0.894212,0.205016,0.059820,0.748718,0.110789,0.877053,0.143336
7,imbalanced,Bayesian_GaussianNB,0.129807,0.520156,0.207765,0.867946,0.163571,0.077838,0.506294,0.134932,0.854416,0.104965



Tốt nhất theo valid PR-AUC (train=balanced_smote): ANN_MLP

Tốt nhất theo valid PR-AUC (train=imbalanced): ANN_MLP


In [14]:
# Confusion matrix + classification report (validation) theo từng train_variant + model
for train_variant in sorted(artifacts.keys()):
    for name in artifacts[train_variant]:
        valid_prob = artifacts[train_variant][name]["valid_prob"]
        y_pred = (valid_prob >= 0.5).astype(int)

        cm = confusion_matrix(y_valid, y_pred)
        print(f"\n[{train_variant}] {name} - Validation confusion matrix")
        print(cm)
        print(f"\n[{train_variant}] {name} - Validation classification report")
        print(classification_report(y_valid, y_pred, digits=4, zero_division=0))



[balanced_smote] LogisticRegression - Validation confusion matrix
[[236610  21187]
 [   355   1183]]

[balanced_smote] LogisticRegression - Validation classification report
              precision    recall  f1-score   support

           0     0.9985    0.9178    0.9565    257797
           1     0.0529    0.7692    0.0990      1538

    accuracy                         0.9169    259335
   macro avg     0.5257    0.8435    0.5277    259335
weighted avg     0.9929    0.9169    0.9514    259335


[balanced_smote] DecisionTree - Validation confusion matrix
[[247044  10753]
 [   133   1405]]

[balanced_smote] DecisionTree - Validation classification report
              precision    recall  f1-score   support

           0     0.9995    0.9583    0.9784    257797
           1     0.1156    0.9135    0.2052      1538

    accuracy                         0.9580    259335
   macro avg     0.5575    0.9359    0.5918    259335
weighted avg     0.9942    0.9580    0.9739    259335


[balanced

In [19]:
# Lưu kết quả so sánh ra CSV + mô hình đã fit (.pkl)
out_dir = ARTIFACT_DIR / "modeling_results"
out_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_dir / "model_comparison.csv", index=False)
print("Saved:", (out_dir / "model_comparison.csv").resolve())

models_dir = out_dir / "trained_models"
models_dir.mkdir(parents=True, exist_ok=True)
for train_variant, by_name in artifacts.items():
    for name, payload in by_name.items():
        safe = f"{train_variant}__{name}".replace(" ", "_")
        path = models_dir / f"{safe}.pkl"
        joblib.dump(payload["fitted_model"], path)
        print("Model:", path.resolve())
print(f"Đã lưu {sum(len(v) for v in artifacts.values())} file .pkl tại: {models_dir.resolve()}")


Saved: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\model_comparison.csv
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\imbalanced__LogisticRegression.pkl
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\imbalanced__DecisionTree.pkl
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\imbalanced__Bayesian_GaussianNB.pkl
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\imbalanced__ANN_MLP.pkl
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\balanced_smote__LogisticRegression.pkl
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\balanced_smote__DecisionTree.pkl
Model: D:\Project\DataMining-DataWarehouse\Source\artifacts\modeling_results\trained_models\balanced_smote__Bayesian_GaussianNB.pkl
Model: D:\Project\Data